In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Using an LLM to improve prompts to an LLM

**References**
- [LLMs are Human-Level Prompt Engineers](https://arxiv.org/pdf/2211.01910.pdf)
    - [APE summary](https://sites.google.com/view/automatic-prompt-engineer)
    
The Automatic Prompt Engineer (APE) is a system to *improve* upon prompts
- given a prompt
- APE will create a prompt that is *more effective*

It uses an LLM for multiple purposes
- to create variations of the given prompt
- to evaluate which variation is best

# Using APE to improve upon Instruction Following

The particular task used to illustrate APE is *instruction following*

Given 
- an instruction for a task
- along with an instance of the task
- create output that solves the task on the input

For example, our Summarization task
    
    {PREFIX} {DOC} Summary: {SUMMARY}

`{PREFIX}` corresponds to the Instruction describing what a `{SUMMARY}` is

APE has been demonstrated to improve prompts 
- **for a specific task** 
- not improvement of general prompts


The particular `{PREFIX}` we use is a combination of
- a Instruction: textual *task description* 
- zero or more exemplars: 
    - demonstrating examples of the input/output relationship described by the instruction
    - In Context Learning  

    {PREFIX} = {INSTRUCTION} {EXEMPLARS}
    
    
Given just the exemplar
- we want APE to *create*  the "best" Instruction

Here is a picture of the workflow.

We explain each step below.

<table>
    <center><strong>APE Workflow</strong></center>
    <tr>
        <img src="images/ape_workflow.png" width=70%>
    </tr>
    
Attribution: https://arxiv.org/pdf/2211.01910.pdf#page=2
</table>

The APE method for automatically generating "good" instructions is conceptually simple.
- the work is performed by "helper" LLMs


As a first step (the "Proposal" in step 1 of the chart)
- we give the helper LLM  the exemplars `{EXEMPLARS}`
- and a *place-holder* `{INSTRUCTION}` for the Instruction
    - place-holder is labeled `<INSERT>` in the chart

and ask the helper to create the Instruction.

We do this several times to generate multiple plausible instructions, conditional on the exemplars.

**Notes**

Rather than "predict the next token" of the suffix (the Language Modeling objective)
- we use a model that knows how to "fill in the blank" (also trained on a Masked Language Modeling objective)

In the next step (the "Scoring" and "Log probability" in steps 2  and 3 of the chart)
- we ask a "helper" LLM to rank the candidate Instructions from the Proposal step
    - create a "score" (log probability)
    - that a candidate instruction is consistent with the Exemplars
    
**Note**
- log probability: less negative is better
- an LLM produces each token from a probability distribution
    - the probability that each token in the vocabulary is appropriate
    - the chaining (multiplying)the probability of each token of the candidate gives the likelihood of the candidate
    
    
    

In an optional step ("High score candidates" and "Similar candidates" in steps 4 and 5 of the chart)
- using only the highly-ranked candidates
- we ask a "helper" LLM to generate variations of the candidate Instruction


## Forward/Reverse mode generation of candidates

This is a minor technical point.

The prompt in Step 1 of the  workflow above is not in the format consistent with text-continuation
- format is called *forward generation*
- so must use an LLM that solves Masked Language task, rather than text-continuation

An alternate prompt can be used that is consistent with text-continuation
- format is called *reverse generation*

<table>
    <center><strong>APE Forward/Reverse Generation templates</strong></center>
    <tr>
        <td>
            <img src="images/ape_fwd_generation.png">
        </td>
        <td>
            <img src="images/ape_reverse_generation.png">
        </td>
    </tr>
    
Attribution: https://arxiv.org/pdf/2211.01910.pdf#page=4
</table>

# APE evaluation: super-human performance !

Here is a comparison of APE generated prompts
- versus
    - an alternate method (previously published), labeled "Greedy"
    - a human engineer (horizontal dotted line)
- evaluated on models of various sizes
    - GPT-3
    - Instruct GPT-3 (fine-tuned for instruction following)
- using 24 NLP tasks

**Note**

The reported statistic is *interquartile mean* (i.e., average after dropping the upper and lower 25% of results)

<table>
    <center><strong>APE Workflow and Results</strong></center>
    <tr>
        <img src="images/ape_results.png" width=70%>
    </tr>
    
Attribution: https://arxiv.org/pdf/2211.01910.pdf#page=2
</table>

# Zero-shot: Improving on "Let's think step by step"

[Chain of Thought (CoT) prompting](NLP_Beyond_LLM.ipynb#Chain-of-thought-prompting)
- is a simple technique
- for create prompts with better performance
- for multi-step reasoning problems

In the few-shot setting
- exemplars demonstrate step by step reasoning
- eliciting the LLM to produce text continuation that also exhibits step by step reasoning

In the zero-shot setting, it simply involves appending
> Let's think step by step

to the prompt

<table>
    <center><strong>Chain of Thought Prompting</strong></center>
    <tr>
        <img src="images/cot_step_by_step.png" width=80%>
    </tr>
    
    Attribution: https://arxiv.org/pdf/2201.11903.pdf
</table>

Let's [use APE](https://arxiv.org/pdf/2211.01910.pdf#page=19) 
to find a zero-shot prompt appendage that improves upon 
>Let's think step by step

That is: the instruction consists of
- the task description
- followed by a "magic suffix" (e.g., "let's think step by step")

We want to find the best "magic suffix".

The authors use the following template (where `INPUT` and `OUTPUT` are place-holders for an actual question and answer pair).

        Instruction: Answer the following question
        
        Q: INPUT
        A: Let's <INSERT> OUTPUT

We are using forward-mode generation to get APE 
- to create a phrase that follows the `INPUT`
- that begins with the word "Let's"
- and is followed by <INSERT> 
    
APE will solve for the best value of <INSERT>.

APE creates
> Let’s work this out in a step by step way to be sure we have the right answer.

and the author's demonstrate improved performance on several benchmarks.

This is a nice demonstration of using an LLM to help craft better prompts to LLM's.

In [2]:
print("Done")

Done
